In [9]:
import nest_asyncio
nest_asyncio.apply()

import warnings
warnings.filterwarnings("ignore")

import os

# Manually parse .env file 
with open(".env", "r") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

uri = os.environ.get("NEO4J_URI")
username = os.environ.get("NEO4J_USERNAME")
password = os.environ.get("NEO4J_PASSWORD")
database = os.environ.get("NEO4J_DATABASE")

print("URI loaded:", uri is not None)
print("Username loaded:", username is not None)
print("Password loaded:", password is not None)
print("Database loaded:", database is not None)

URI loaded: True
Username loaded: True
Password loaded: True
Database loaded: True


In [13]:
#NEO4j CONNECTION TEST
# Run this to confirm credentials are working before running the full pipeline
from neo4j import GraphDatabase
driver = GraphDatabase.driver(uri, auth=(username, password))

try:
    with driver.session() as session:
        result = session.run("RETURN 'Connected to Neo4j AuraDB!' AS message")
        for record in result:
            print(record["message"])
    print("Connection successful!")
except Exception as e:
    print("Connection failed:", e)
finally:
    driver.close()

Connected to Neo4j AuraDB!
Connection successful!


In [14]:
#OLLAMA TEST
from llama_index.llms.ollama import Ollama

llm = Ollama(model="llama3:8b", request_timeout=120.0)

response = llm.complete("Say hello and confirm you are running locally.")
print(response)

Hello! I'm here!

Just to confirm, I am indeed running locally on your device, handling our conversation in real-time. I don't have any external dependencies or connections, so everything is happening right here where we're chatting. How can I help you today?


In [15]:
#PDF EXTRACTION
import pdfplumber
import os

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    return text

pdf_files = [f for f in os.listdir("data") if f.endswith(".pdf")]
print(f"PDF files found: {pdf_files}")

if pdf_files:
    text = extract_text_from_pdf(f"data/{pdf_files[0]}")
    print(f"\nExtracted text preview:")
    print(text[:500])

PDF files found: ['MathsComp_Lecture1.pdf', 'MathsComp_Lecture10.pdf', 'MathsComp_Lecture11.pdf', 'SeminarTasks12_MathComp.pdf', 'SeminarTasks3_MathComp.pdf', 'SeminarTasks4_MathComp.pdf', 'MathsComp_Lecture2.pdf', 'MathComp-Module-Proforma-Sep2024.pdf', 'MathsComp_Lecture3.pdf', 'MathsComp_Lecture7.pdf', 'MathsComp_Lecture4.pdf', 'MathsComp_Lecture5.pdf', 'SeminarTasks2_MathComp.pdf', 'ICT-Revision-Session-November2025.pdf', 'SeminarTasks9_MathComp.pdf', 'MathsComp-Q&ASession7.pdf', 'SeminarTasks11_MathComp.pdf', 'MathsComp_Lecture8.pdf', 'MathsComp_Lecture9.pdf', 'MathComp Schedule SEP2025 StudentsVersion.pdf', 'SeminarTasks0_MathComp.pdf', 'SeminarTasks7_MathComp.pdf', 'SeminarTasks10_MathComp.pdf', 'SeminarTasks1_MathComp.pdf']

Extracted text preview:
4COSC002W Mathematics
for Computing
Lecture 1
Module Introduction. Number Formats. Types of Numbers.
Number Theory Basics. Modular Arithmetic. Sequences.
WELCOME
TO THE MATHS FOR COMPUTING MODULE!
Module Leader:
Dr Natalia Yerashenia

In [16]:
# Extract text from all PDFs
all_documents = {}

for pdf_file in pdf_files:
    text = extract_text_from_pdf(f"data/{pdf_file}")
    all_documents[pdf_file] = text
    print(f"✓ {pdf_file}: {len(text)} characters extracted")

print(f"\nTotal documents processed: {len(all_documents)}")

✓ MathsComp_Lecture1.pdf: 18667 characters extracted
✓ MathsComp_Lecture10.pdf: 8939 characters extracted
✓ MathsComp_Lecture11.pdf: 7742 characters extracted
✓ SeminarTasks12_MathComp.pdf: 1390 characters extracted
✓ SeminarTasks3_MathComp.pdf: 6625 characters extracted
✓ SeminarTasks4_MathComp.pdf: 4945 characters extracted
✓ MathsComp_Lecture2.pdf: 10920 characters extracted
✓ MathComp-Module-Proforma-Sep2024.pdf: 14296 characters extracted
✓ MathsComp_Lecture3.pdf: 15637 characters extracted
✓ MathsComp_Lecture7.pdf: 14708 characters extracted
✓ MathsComp_Lecture4.pdf: 8091 characters extracted
✓ MathsComp_Lecture5.pdf: 12714 characters extracted
✓ SeminarTasks2_MathComp.pdf: 7713 characters extracted
✓ ICT-Revision-Session-November2025.pdf: 1484 characters extracted
✓ SeminarTasks9_MathComp.pdf: 3649 characters extracted
✓ MathsComp-Q&ASession7.pdf: 3537 characters extracted
✓ SeminarTasks11_MathComp.pdf: 2265 characters extracted
✓ MathsComp_Lecture8.pdf: 13807 characters extract

In [17]:
# EXTRACTED TEXT INTO LlamaIndex DOCUMENT OBJECTS
from llama_index.core import Document
llama_documents = []

for filename, text in all_documents.items():
    if text.strip():  # only add if text is not empty
        doc = Document(
            text=text,
            metadata={"filename": filename}
        )
        llama_documents.append(doc)

print(f"Documents ready for pipeline: {len(llama_documents)}")
print(f"\nExample document metadata: {llama_documents[0].metadata}")
print(f"Example text preview: {llama_documents[0].text[:200]}")

Documents ready for pipeline: 24

Example document metadata: {'filename': 'MathsComp_Lecture1.pdf'}
Example text preview: 4COSC002W Mathematics
for Computing
Lecture 1
Module Introduction. Number Formats. Types of Numbers.
Number Theory Basics. Modular Arithmetic. Sequences.
WELCOME
TO THE MATHS FOR COMPUTING MODULE!
Mod


In [18]:
#PIPELINE SETUP
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from typing import Literal

# Set up local LLM
llm = Ollama(model="llama3:8b", request_timeout=300.0)

# Set up local embedding model (free, no API needed)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Define the schema — what we want to extract
# This should eventually be defined jointly with O4
entities = Literal["CONCEPT", "TOPIC", "WEEK", "ASSESSMENT", "PREREQUISITE"]

relations = Literal[
    "PREREQUISITE_OF",  # concept A must be learned before concept B
    "PART_OF",          # concept belongs to a topic or week
    "RELATED_TO",       # concepts are related
    "ASSESSED_BY",      # concept is assessed by an assessment
    "INTRODUCED_IN"     # concept is first introduced in a week
]

validation_schema = [
    ("CONCEPT", "PREREQUISITE_OF", "CONCEPT"),
    ("CONCEPT", "PART_OF", "TOPIC"),
    ("CONCEPT", "PART_OF", "WEEK"),
    ("TOPIC", "PART_OF", "WEEK"),
    ("CONCEPT", "ASSESSED_BY", "ASSESSMENT"),
    ("CONCEPT", "INTRODUCED_IN", "WEEK"),
    ("CONCEPT", "RELATED_TO", "CONCEPT"),
]

# Set up the extractor
kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    possible_entities=entities,
    possible_relations=relations,
    kg_validation_schema=validation_schema,
    strict=True
)

print("Pipeline components ready:")
print("✓ LLM: Llama-3 8B (local)")
print("✓ Embeddings: BAAI/bge-small-en-v1.5 (local)")
print("✓ Schema: defined with", len(validation_schema), "relationship types")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5194.72it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline components ready:
✓ LLM: Llama-3 8B (local)
✓ Embeddings: BAAI/bge-small-en-v1.5 (local)
✓ Schema: defined with 7 relationship types


In [ ]:
#BUILD KNOWLEDGE GRAPH
from llama_index.core import PropertyGraphIndex
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

# Fresh store every time
fresh_store = Neo4jPropertyGraphStore(
    username=username,
    password=password,
    url=uri,
    database=database
)
# Start with just Lecture 1 for testing
test_document = [doc for doc in llama_documents 
                 if doc.metadata["filename"] == "MathsComp_Lecture1.pdf"]

print(f"Documents selected: {len(test_document)}")
print(f"Processing: {test_document[0].metadata['filename']}")

# Build the graph in AuraDB
index = PropertyGraphIndex.from_documents(
    test_document,
    kg_extractors=[kg_extractor],
    embed_model=embed_model,
    property_graph_store=fresh_store,
    show_progress=True
)


print("\nKnowledge graph built successfully!")

Documents selected: 1
Processing: MathsComp_Lecture1.pdf


Generating embeddings: 100%|██████████| 3/3 [00:00<00:00,  7.61it/s]



Knowledge graph built successfully!


In [23]:
import nest_asyncio
nest_asyncio.apply()
from llama_index.core import Settings

# Setting LlamaIndex to use Ollama globally
Settings.llm = llm
Settings.embed_model = embed_model

# Set up query engine with explicit LLM
query_engine = index.as_query_engine(
    llm=llm,
    include_text=True,
    response_mode="tree_summarize"
)

# Ask a question about Lecture 1 content
response = query_engine.query(
    "What are the main mathematical concepts in Lecture 1 and how are they related?"
)

print(response)

The main mathematical concepts in Lecture 1 include Number Theory, Integers, Prime Numbers, Composite Numbers, Factorisation, Division Theorem, Geometric Sequence, Fibonacci Sequence, Arithmetic Sequence, and Binary Number System. These concepts are all interrelated within the context of number theory and its applications.

Number Theory is a branch of mathematics that deals with the study of integers and other whole numbers. Integers themselves include Prime Numbers and Composite Numbers, which are fundamental building blocks for understanding factorisation and division. Factorisation refers to representing an integer as a product of prime numbers, while Division Theorem provides a way to decompose a number into its prime factors.

Geometric Sequence is related to the Fibonacci Sequence, which is a series of numbers that follows a specific pattern. Arithmetic Sequence is another type of sequence that involves adding a constant difference between consecutive terms.

Binary Number Syste

In [25]:
nest_asyncio.apply()

response = query_engine.query(
    "What is modular arithmetic and what are its applications?"
)
print(response)

Modular Arithmetic operates on numbers with a fixed range, much like how binary numbers wrap around when adding or subtracting (mod2). It is closely related to the Binary Number System.
